# TrustGuard — prediction notebook (Track 2)
**This is the file uploaded to the portal (card 1), next to `model.pkl` (card 2).**
It follows the organisers' `template_notebook.ipynb`: the sandbox runs it top-to-bottom with no internet and
no `train.csv`; the test file (`DATATHON_INPUT_PATH`) has **no `id` column and shuffled rows**; we write one
probability per row, same order, under a single `prediction` column to `DATATHON_OUTPUT_PATH`.
One upload per 2h and a failed run burns the window — always dry-run locally first.

The feature code below is inlined from `notebooks/submission.ipynb` §3–4 and must stay byte-identical to it.
`model.pkl` must be trained under the platform's versions (`.venv-image`, from `requirements-image.txt`).


In [ ]:
import os
import joblib
import pandas as pd

TEST_PATH = os.environ.get("DATATHON_INPUT_PATH", "test.csv")
OUTPUT_PATH = os.environ.get("DATATHON_OUTPUT_PATH", "predictions.csv")


In [ ]:
# --- feature engineering: keep byte-identical to the training notebook (§3-4) ---
MERCHANTS = ["cash_transfer", "electronics", "entertainment", "food_delivery", "gaming",
             "grocery", "luxury", "retail", "transport", "travel", "utilities"]
COUNTRIES = ["AU", "GB", "ID", "JP", "MY", "PH", "SG", "TH", "US", "VN"]
CHANNELS  = ["bank_transfer", "card_present", "ecommerce", "mobile_app"]
CAT_VOCAB = {"merchant_category": MERCHANTS, "country": COUNTRIES, "transaction_channel": CHANNELS}
FEATURES = ["transaction_amount", "transaction_hour", "merchant_category", "country",
            "transaction_channel", "transactions_last_24h", "spend_last_24h",
            "account_age", "new_device", "transactions_last_1h"]

def clean(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for c, vocab in CAT_VOCAB.items():
        out[c] = pd.Categorical(out[c], categories=vocab)
    return out

def make_features(df: pd.DataFrame) -> pd.DataFrame:
    out = clean(df)
    return out[FEATURES]


In [ ]:
model, features = joblib.load("model.pkl")
test_df = pd.read_csv(TEST_PATH)  # sandbox: no id column, shuffled rows — features never use id
X_test = make_features(test_df)[features]
if hasattr(model, "predict_proba"):
    preds = model.predict_proba(X_test)[:, 1]
else:  # lightgbm Booster: predict() already returns P(fraud)
    preds = model.predict(X_test)

assert len(preds) == len(test_df)
assert 0 <= preds.min() <= preds.max() <= 1
pd.DataFrame({"prediction": preds}).to_csv(OUTPUT_PATH, index=False)
print(f"wrote {len(preds)} predictions to {OUTPUT_PATH}, mean p {preds.mean():.4f}")
